# Dubai Real Estate 2025 - Developer Extraction
This notebook engineers a developer feature for Dubai's 2025 residential real estate transactions. Developer information is not provided as a dedicated field in the original dataset and must therefore be inferred from the available text fields Project, Building and Master Project.

Developer identification is treated as a semi-automated entity-resolution problem. The pipeline combines a curated developer dictionary, exact text matching, manually validated mappings, developer-name standardization and similarity-based propagation. Candidate assignments are reviewed conservatively, prioritizing precision over maximum coverage where the available text is ambiguous.

master_project is retained as contextual information for manual validation but excluded from automated exact matching because it frequently represents a master development or geographical location rather than the property developer. Preliminary validation showed that interpreting developer-like names in this field directly could generate false assignments.

## Main Steps

**- Data Preparation:** Relevant building, project and master_project information is normalized while preserving the original transaction identifier for final integration.  

**- Exact Developer Matching:** A curated dictionary of verified developer names is matched against normalized project and building text to establish a high-confidence initial set of developer assignments.

**- Manual Mapping:** High-frequency unresolved combinations of master project, project, and building information are manually researched and mapped where sufficient evidence is available.

**- Developer Name Standardization:** Equivalent developer labels are consolidated into consistent canonical names, while potentially similar but distinct developers are retained separately. 

**- Similarity-Based Propagation:** Developer assignments are propagated from resolved to unresolved project and building names where strong lexical similarity is observed. Candidate matches are manually reviewed and ambiguous cases explicitly excluded.

**- Prefix-Based Similarity Propagation:** Project-name prefixes are used to recover additional related developments whose full names differ because of phase, tower or other identifying information. Building-prefix propagation is evaluated but rejected because validation produces predominantly ambiguous matches.

**- Final Integration:** The resulting developer feature is merged back into the main analytical dataset through the unique transaction_id and exported for subsequent analysis and predictive modelling.

## Outcome

The final pipeline identifies a developer for approximately **63.23% of residential transactions.**

This coverage is intentionally lower than earlier experimental versions of the extraction procedure, which reached approximately 69% but relied on broader matching rules that were subsequently found to introduce false-positive developer assignments. The final methodology therefore favors a more conservative and auditable balance between coverage and assignment reliability.

As an exploratory validation, a separate preliminary model showed that adding the engineered developer feature increased adjusted R² from approximately 0.50 to 0.63, while a basic Random Forest assigned approximately 20% feature importance to the developer variable. These tests were conducted separately for methodological evaluation and are not included in this notebook.

The predictive value of the feature will be evaluated more rigorously during the modelling stage of the project. The resulting developer variable provides a structured feature for investigating whether developer identity contributes explanatory and predictive information beyond property characteristics and location.

# Libraries

In [3]:
import pandas as pd # Data Manipulation
import numpy as np # Data Manipulation

import re
from rapidfuzz import process, fuzz
from itertools import combinations

# Data

In [5]:
raw = pd.read_csv('re_2025_analysis.csv')
df = raw.copy()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220918 entries, 0 to 220917
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        220918 non-null  int64  
 1   transaction_id    220918 non-null  object 
 2   transaction_type  220918 non-null  object 
 3   date              220918 non-null  object 
 4   property_type     220918 non-null  object 
 5   registration      220918 non-null  object 
 6   area              220918 non-null  object 
 7   building          194538 non-null  object 
 8   project           197958 non-null  object 
 9   master_project    194085 non-null  object 
 10  landmark          220918 non-null  object 
 11  metro             220918 non-null  object 
 12  mall              220918 non-null  object 
 13  rooms             220918 non-null  object 
 14  parking           220918 non-null  int64  
 15  size              220918 non-null  float64
 16  price             22

# 1. Data Preparation
The three text fields relevant to developer extraction are preprocessed to ensure consistent string-based matching.

## 1.1. Creating Subset
A subset containing the Transaction_ID, Project, Building and Master Project fields is created for the developer-extraction workflow. Master Project is retained for contextual inspection but is excluded from the developer-matching procedures because it may represent a location or master development rather than the developer of the individual property.

In [9]:
# Subset and copy (avoid modifying original)
df_dev = df[['transaction_id', 'building', 'project', 'master_project']].copy()

## 1.2. Normalize Null Values
The Project, Building and Master Project fields contain missing values that were intentionally preserved in the previous data-preparation notebook because they are relevant to the developer-extraction process. Here, missing values in these three text fields are converted to empty strings to facilitate subsequent string-based matching.

In [11]:
df_dev = df_dev.fillna('').astype(str)

## 1.3. Normalize Text
Basic text normalization is applied to the three source fields to standardize capitalization and whitespace and improve consistency during entity matching.

In [13]:
# --- Normalize all text ---
def normalize_text(s):
    """
    Clean and standardize strings for text matching.
    - Convert to lowercase
    - Remove excessive whitespace
    - Strip leading/trailing spaces
    """
    s = str(s).lower().strip()
    s = re.sub(r'\s+', ' ', s)       # collapse multiple spaces
    return s

In [14]:
for col in ['building', 'project', 'master_project']:
    df_dev[col] = df_dev[col].apply(normalize_text)

# 2. Exact Developer Matching
A curated list of known developers operating in Dubai was compiled from external sources. Each developer name is searched independently within the Building and Project fields.

The two fields are treated as independent sources rather than assigning priority to one over the other. When only one match is found, that match is retained. Cases where two different developers are matched are treated as conflicts and manually validated.

A coverage of **32.58%** was achieved during this process.

## 2.1. Developer Dictionary and Matching Pattern

In [17]:
# List of Dubai known developers 
developers = [
    'aark developers',  'acube', 'al ghurair',
    'al habtoor', 'al helal al zahaby', 'aldar', 'amana',
    'aqua properties', 'arada', 'arista', 'artar',
    'asak real estate development', 'aurora', 'avenew development',
    'azizi', 'beyond', 'binghatti', 'bnh', 'b&h',
    'burtville', 'centurion', 'citi developers', 
    'condor', 'confident group', 'damac', 'danube',
    'dar al arkan', 'dar global', 'dec', 'deyaar',
    'diamond developers', 'digo', 'dubai properties', 'durar',
    'eagle hills', 'ellington', 'emaar', 'empire developments',
    'esnaad', 'fakhruddin', 'five holdings', 'gfs', 'green group',
    'green horizon','grovy', 'gulf general investment', 'h&h',
    'heilbronn', 'hmb', 'igo', 'iman', 'imtiaz', 
    'irth', 'jumeirah golf estates', 'karma', 'kasco',
    'khamas group', 'leos', 'lincoln star',
    'lmd', 'london gate', 'lootah', 'maas',
    'mag', 'majid', 'majid al futtaim', 'manazil',
    'meraas', 'metac', 'meteora', 'nakheel', 'naseeb group', 
    'national bonds', 'national properties', 'nshama',
    'object 1', 'octa', 'omniyat', 'one development',
    'orange life', 'oro24', 'pantheon', 'peace homes', 
    'prescott', 'prestige one', 'refine', 'reportage', 
    'royal development company', 'saas', 'samana', 
    'select group', 'seven tides', 'shamal', 'sobha', 
    'sol properties', 'sol development','stamn', 'sunrise capital',
    'symbolic', 'tabeer', 'taraf', 'tiger',
    'time properties', 'townx', 'trident', 'union properties',
    'vincitore', 'vision', 'wadan', 'wasl',
    'zaya', 'zazen', 'zimaya'
]

In [18]:
text_columns = ['building', 'project']

In [19]:
# Create Regex pattern to search for developers
pattern_exact = re.compile(
    r'\b(' + '|'.join(map(re.escape, developers)) + r')\b'
)

## 2.2. Developer Extraction

In [21]:
# Extracting developers from the two columns
def extract_all_developers(text):
    """
    Return all unique developer names found in a text string.
    """
    matches = [m.group(1) for m in pattern_exact.finditer(text)]
    
    # Preserve order while removing duplicates
    return list(dict.fromkeys(matches))

for col in ['building', 'project']:
    df_dev[f'{col}_exact'] = df_dev[col].apply(extract_all_developers)

In [22]:
def combine_matches(row):
    matches = (
        row['building_exact']
        + row['project_exact']
    )
    
    return list(dict.fromkeys(matches))

In [23]:
df_dev['developer_candidates'] = df_dev.apply(
    combine_matches,
    axis=1
)

In [24]:
df_dev['n_developer_candidates'] = (
    df_dev['developer_candidates'].str.len()
)

In [25]:
df_dev['n_developer_candidates'].value_counts().sort_index()

n_developer_candidates
0    148937
1     71800
2       181
Name: count, dtype: int64

## 2.3. Conflict Validation and Resolution
The exact-matching procedure identified 71,800 transactions with a single developer candidate and 181 transactions containing two different candidates.

Single-candidate matches are assigned directly to 'developer_exact'.

Inspection of 'building_exact', 'project_exact' and 'developer_candidates' shows that all 181 multi-candidate records correspond to the same candidate pair: Binghatti / Aurora. Manual validation against external sources confirms that Binghatti is the property developer, while Aurora refers to the building/project name in these transactions.

Because this resolution is specific to the present dataset, it is applied explicitly rather than incorporated into the general developer-assignment function. This step should be reassessed if the methodology is applied to another dataset.

After resolving the ambiguous records, exact developer matching achieves **32.58% coverage.**

In [27]:
conflicts_exact = df_dev[
    df_dev['n_developer_candidates'] > 1
].copy()

conflicts_exact['developer_candidates'].value_counts()

developer_candidates
[binghatti, aurora]    181
Name: count, dtype: int64

In [28]:
conflicts_exact['project_exact'].value_counts()

project_exact
[binghatti, aurora]    181
Name: count, dtype: int64

In [29]:
conflicts_exact['building_exact'].value_counts()

building_exact
[binghatti, aurora]    181
Name: count, dtype: int64

In [30]:
def inspect_exact_conflict(dev1, dev2, n=20):
    mask = df_dev['developer_candidates'].apply(
        lambda x: set(x) == {dev1, dev2}
    )

    result = df_dev.loc[
        mask,
        [
            'building',
            'project',
            'master_project',
            'building_exact',
            'project_exact',
            'developer_candidates'
        ]
    ].copy()

    # Convert list columns to tuples so pandas can group/hash them
    for col in [
        'building_exact',
        'project_exact',
        'developer_candidates'
    ]:
        result[col] = result[col].apply(tuple)

    result = (
        result
        .value_counts(
            subset=[
                'building',
                'project',
                'master_project',
                'building_exact',
                'project_exact',
                'developer_candidates'
            ]
        )
        .reset_index(name='transactions')
        .sort_values('transactions', ascending=False)
        .reset_index(drop=True)
    )

    return result.head(n)

In [31]:
inspect_exact_conflict('binghatti', 'aurora')

,building,project,master_project,building_exact,project_exact,developer_candidates,transactions
0,binghatti aurora,binghatti aurora,jumeirah village circle,"(binghatti, aurora)","(binghatti, aurora)","(binghatti, aurora)",181


In [32]:
def assign_developer(candidates):
    """
    Assign unambiguous exact matches.
    Rows with zero or multiple candidates remain unresolved.
    """
    if len(candidates) == 1:
        return candidates[0]
    else:
        return np.nan

In [33]:
df_dev['developer_exact'] = df_dev['developer_candidates'].apply(
    assign_developer
)

In [34]:
df_dev.loc[
    df_dev['developer_candidates'].apply(
        lambda x: x == ['binghatti', 'aurora']
    ),
    'developer_exact'
] = 'binghatti'

In [35]:
print("Exact-match coverage:",
      f"{df_dev['developer_exact'].notna().mean()*100:.2f}% of records")

Exact-match coverage: 32.58% of records


# 3. Manual Mapping
Exact developer matching provides a high-precision starting point, but a substantial proportion of transactions remain unmatched because developer names are not consistently included in the available text fields.

To improve coverage, high-frequency unmatched combinations of Master Project, Project and Building were manually reviewed using external sources. Unlike the automated exact-matching stage, master_project is included here only as contextual information within a composite lookup key; it is not independently interpreted as the property developer.

After applying manual mapping, developer matching achieves **58.27% coverage.**

## 3.1. Identify High-Frequency Unmatched Combinations
A composite proj_concat identifier is created by combining the normalized master_project, project and building fields. This provides additional context for identifying developments where individual fields may contain incomplete or ambiguous information.

Only transactions that remain unresolved after exact matching are considered. Their unique proj_concat values are ranked by transaction frequency so that manual validation focuses first on combinations affecting the largest number of records.

The resulting high-frequency combinations were manually researched and assigned to a developer where sufficient evidence was available. This review produced a curated developer mapping used in the following step.

In [38]:
# --- Combine all 3 columns into a single searchable text field ---
# Starting with master_project
df_dev['proj_concat'] = (
    df_dev['master_project'] + ' | ' +
    df_dev['project'] + ' | ' +
    df_dev['building']
).str.strip()

In [39]:
# Identify the most frequent combinations still unmatched after exact matching
unmatched = df_dev[df_dev['developer_exact'].isna()]

top_unmatched = (
    unmatched['proj_concat']
    .value_counts()
    .rename_axis('proj_concat')
    .reset_index(name='count')
)

top_unmatched.head()

,proj_concat,count
0,| |,894
1,arabian ranches - 1 | |,827
2,dubai science park | skyhills astra | sky hill...,701
3,dubai marina | rove home dubai marina | rove h...,655
4,| skyvue | skyvue solair,648


## 3.2. Apply Curated Manual Mapping
The manually validated mappings are stored in developer_manual_mapping_final.csv and merged back into the developer-extraction dataset using proj_concat as the lookup key.

Exact-match assignments retain priority. Manual assignments are therefore used only where developer_exact remains missing.

After incorporating the manually validated mappings, **developer coverage increases from 32.58% to 58.27%**, representing an improvement of approximately 25.7 percentage points over exact matching alone.

The remaining unmatched transactions are retained for subsequent similarity-based developer resolution.

In [41]:
developer_manual_mapping = pd.read_csv('developer_manual_mapping_final.csv').drop(columns=['count'], errors='ignore')

In [42]:
df_dev = df_dev.merge(
    developer_manual_mapping,
    on='proj_concat',
    how='left'
)

In [43]:
df_dev['developer'] = (
    df_dev['developer_exact']
    .fillna(df_dev['developer_manual'])
)

In [44]:
exact_coverage = df_dev['developer_exact'].notna().mean() * 100
combined_coverage = df_dev['developer'].notna().mean() * 100

print(f"Exact-match coverage: {exact_coverage:.2f}%")
print(f"Combined coverage: {combined_coverage:.2f}%")
print(
    f"Incremental coverage from manual mapping: "
    f"{combined_coverage - exact_coverage:.2f} pp"
)

Exact-match coverage: 32.58%
Combined coverage: 58.27%
Incremental coverage from manual mapping: 25.68 pp


# 4. Developer Name Standardization
Following exact matching and manual mapping, developer labels are reviewed for naming inconsistencies before being used in subsequent similarity-based propagation.

Different naming conventions may refer to the same underlying developer, for example when company descriptors such as Properties, Development or Group are appended to an otherwise identical developer name. Leaving these variants untreated would artificially increase the number of unique developers and fragment transactions belonging to the same developer across multiple labels.

For the purposes of this analysis, such variants are consolidated under a single, concise canonical developer name. The objective is to represent the developer identity consistently rather than distinguish between related corporate entities or naming variants.

## 4.1. Contained Developer Names
Developer names are first compared to identify cases where one label is fully contained within another. This check identifies 19 potential pairs.

Manual inspection confirms that these pairs represent naming variants of the same developer, generally resulting from additional company descriptors such as Properties, Development, or Group. These variants are therefore consolidated under the shorter canonical developer name.

In [47]:
dev_names = sorted(df_dev['developer'].dropna().unique())
pairs_contained = []

for i, a in enumerate(dev_names):
    for b in dev_names[i+1:]:
        # skip if identical
        if a == b: 
            continue
        # check containment 
        if a in b or b in a:
            pairs_contained.append((a, b, len(a)/len(b) if len(b)>0 else 0))

contain_df = pd.DataFrame(pairs_contained, columns=['dev1','dev2','length_ratio'])
display(contain_df.head(30))
print(f"Found {len(contain_df)} containment pairs.")

,dev1,dev2,length_ratio
0,aa and hmb real estate development,hmb,11.333333
1,bnh,bnh developer,0.230769
2,ellington,ellington properties,0.450000
3,fakhruddin,fakhruddin properties,0.476190
4,hmb,hmb homes real estate development,0.090909
5,iman,iman developers,0.266667
6,irth,irth group,0.400000
7,mag,mag,0.750000
8,majid,majid al futtaim,0.312500
9,mashriq elite,mashriq elite development,0.520000


Found 19 containment pairs.


## 4.2. Similar Developer Names
As an additional validation step, fuzzy string matching is used to identify developer names that are highly similar despite not having a direct containment relationship.

The procedure identifies 13 potentially similar developer pairs. Following manual inspection, none are found to represent duplicate developer identities; the similarities are lexical rather than evidence of inconsistent naming.

No additional standardization is therefore required from this check.

In [49]:
dev_unique = sorted([d.strip().lower() for d in df_dev['developer'].dropna().unique()])

# Create a dataframe of all pairs with similarity score
pairs = []
for a, b in combinations(dev_unique, 2):
    score = fuzz.token_sort_ratio(a, b)
    if score >= 85: 
        pairs.append((a, b, score))

dev_pairs = pd.DataFrame(pairs, columns=['developer_1','developer_2','similarity']).sort_values('similarity', ascending=False)

print(f"Found {len(dev_pairs)} potentially duplicated developer pairs.")
display(dev_pairs.head(30))

Found 13 potentially duplicated developer pairs.


,developer_1,developer_2,similarity
6,mag,mag,100.000000
12,tiger properties,time properties,90.322581
3,h&h development,hz development,89.655172
4,hre development,hz development,89.655172
9,saba properties,sbk properties,89.655172
0,ag properties,dhg properties,88.888889
7,one development,townx development,87.500000
2,h&h development,hre development,86.666667
5,hre development,one development,86.666667
8,refine development,serene developments,86.486486


## 4.3. Developer Standardization
The validated naming variants are mapped to their corresponding canonical developer names.

This produces a standardized developer feature that preserves the existing transaction coverage while reducing artificial fragmentation caused by inconsistent naming conventions. The standardized values are used throughout the subsequent developer-resolution stages.

In [51]:
merge_map = {
    'aa and hmb real estate development': 'hmb',
    'bnh developer': 'bnh',
    'hmb homes real estate development':'hmb',
    'ellington properties':'ellington',
    'fakhruddin properties':'fakhruddin',
    'iman developers': 'iman',
    'irth group': 'irth',
    'mag ':'mag',
    'majid': 'majid al futtaim',
    'mashriq elite development': 'mashriq elite',
    'nshama development': 'nshama',
    'prescott real estate development': 'prescott',
    'reportage properties': 'reportage',
    'tabeer developments': 'tabeer',
    'taraf developments': 'taraf',
    'union': 'union properties',
    'national bonds corporation': 'national bonds',
    'tiger group': 'tiger',
    'tiger properties': 'tiger'
}

In [52]:
df_dev['developer'] = df_dev['developer'].replace(merge_map)

# 5. Similarity-Based Propagation
Developer assignments are propagated to currently unmatched projects when their normalized project names closely resemble projects whose developers have already been identified.

Similarity is measured using RapidFuzz's token-sort ratio. Candidate matches are reviewed manually before propagation to prevent similarly named but distinct projects from being assigned to the same developer.

Several similarity thresholds were evaluated during development. A minimum similarity score of 88 was retained for project matching and 90 for building matching, representing the lowest threshold at which additional valid matches could be identified. Candidate matches were manually reviewed, and similarly named but distinct projects were explicitly excluded from propagation.

Validated matches are then used to propagate the developer from the corresponding known project to currently unmatched transactions.

The same procedure is subsequently applied to building names.

**Results:**  
After project-based similarity propagation, overall **developer coverage increases from 58.27% to 61.76%**.  
After building-based similarity propagation, overall **developer coverage increases to 61.96%**.

## 5.1. Projects
**Known project-family rule:** Manual inspection of iterative similarity matches showed that multiple unmatched projects following the naming pattern The Valley - ... belong to Emaar. These records are therefore assigned directly to Emaar before fuzzy project-name propagation. Encoding this validated project-family relationship explicitly avoids relying on chained similarity matches to recover the same information.

In [55]:
valley_mask = (
    df_dev['developer'].isna()
    & df_dev['project'].str.contains(
        r'^the valley\s*-\s*',
        case=False,
        na=False,
        regex=True
    )
)

df_dev.loc[valley_mask, 'developer'] = 'emaar'

In [56]:
def find_similarity_matches(df, field, cutoff):
    """
    Find the closest known value for each unmatched value
    using token-sort similarity.
    """

    known = (
        df[df['developer'].notna()]
        [[field, 'developer']]
        .dropna()
        .drop_duplicates(subset=field)
    )

    unknown = (
        df[df['developer'].isna()]
        [[field]]
        .dropna()
    )

    unknown = unknown[
        unknown[field].str.strip() != ''
    ].drop_duplicates(subset=field)

    matches = []

    for value in unknown[field]:
        match = process.extractOne(
            value,
            known[field],
            scorer=fuzz.token_sort_ratio,
            score_cutoff=cutoff
        )

        if match:
            matched_value, score, _ = match

            matched_dev = known.loc[
                known[field] == matched_value,
                'developer'
            ].iloc[0]

            matches.append({
                f'unmatched_{field}': value,
                f'matched_{field}': matched_value,
                'developer_suggested': matched_dev,
                'similarity': score
            })

    columns = [
        f'unmatched_{field}',
        f'matched_{field}',
        'developer_suggested',
        'similarity'
    ]

    result = pd.DataFrame(matches, columns=columns)

    if not result.empty:
        result = (
            result
            .sort_values('similarity', ascending=False)
            .reset_index(drop=True)
        )

    return result

In [57]:
def apply_similarity_matches(
    df,
    matches,
    field,
    excluded_values=None
):
    if excluded_values is None:
        excluded_values = set()

    unmatched_col = f'unmatched_{field}'

    propagate_map = (
        matches.loc[
            ~matches[unmatched_col].isin(excluded_values)
        ]
        .set_index(unmatched_col)['developer_suggested']
        .to_dict()
    )

    df['developer'] = df['developer'].fillna(
        df[field].map(propagate_map)
    )

    return df

In [58]:
project_matches = find_similarity_matches(
    df_dev,
    field='project',
    cutoff=88
)

print(
    f"{len(project_matches)} unmatched projects have "
    f"similarity >= 88 with known projects."
)

display(project_matches.head())

103 unmatched projects have similarity >= 88 with known projects.


,unmatched_project,matched_project,developer_suggested,similarity
0,forte,forte,emaar,100.0
1,south square,south square,dubai south,100.0
2,expo city sidr residences,expo city sidr residences,expo city,100.0
3,casa flores and eden apartments,casa flores and eden apartments,national properties,100.0
4,bay grove residences c - dubai islands,bay grove residences c - dubai islands,nakheel,100.0


In [59]:
# Exclusion list
excluded_projects = [
    'balqis residence', 'elle residences', 'laya residences',
    'olivia residences', 'liv residence', 'maya townhouses',
    'verdana 2', 'myka residence', 'aria', 'mr.c residences downtown',
    'elevia residences', 'azha downtown residences', 'riva residence',
    'bv residences', 'belmont residences', 'lua residences', 'nb residences'
]

In [60]:
df_dev = apply_similarity_matches(
    df_dev,
    project_matches,
    field='project',
    excluded_values=excluded_projects
)

# verify coverage after propagation
coverage = df_dev['developer'].notna().mean() * 100
print(f"Developer coverage after safe propagation: {coverage:.2f}%")

Developer coverage after safe propagation: 61.76%


## 5.2. Buildings

In [62]:
building_matches = find_similarity_matches(
    df_dev,
    field='building',
    cutoff=90
)

print(
    f"{len(building_matches)} unmatched buildings have "
    f"similarity >= 90 with known buildings."
)

display(building_matches)

15 unmatched buildings have similarity >= 90 with known buildings.


,unmatched_building,matched_building,developer_suggested,similarity
0,cleopatra tower,cleopatra tower,al ghurair,100.000000
1,elite residence 1,elite residence,tameer holdings,93.750000
2,abbey crescent 2,abbey crescent 1,union properties,93.750000
3,goldcrest views,goldcrest views 2,star giga establishment,93.750000
4,hz residences 4,hz residences 5,al helal al zahaby,93.333333
5,barton house 2,barton house 1,union properties,92.857143
6,elite residences 3,elite residence,tameer holdings,90.909091
7,elite residences 2,elite residence,tameer holdings,90.909091
8,elite residences 4,elite residence,tameer holdings,90.909091
9,riviera residence,viera residences,vantage ventures,90.909091


In [63]:
# Exclusion list
excluded_buildings = [
    'riviera residence', 'palma residences', 'amalia residences',
    'dana tower'
]

In [64]:
df_dev = apply_similarity_matches(
    df_dev,
    building_matches,
    field='building',
    excluded_values=excluded_buildings
)

# verify coverage after propagation
coverage = df_dev['developer'].notna().mean() * 100
print(f"Developer coverage after safe propagation: {coverage:.2f}%")

Developer coverage after safe propagation: 61.96%


# 6. Prefix-Based Similarity Propagation
Following full-name similarity propagation, some transactions remain unmatched despite sharing a common naming structure with already identified developments.

To capture these cases, project names are reduced to a more stable prefix by splitting each name at the first numeric or special-character separator. These prefixes are then compared using fuzzy string similarity against prefixes associated with an already identified developer.

Known prefixes associated with more than one developer are excluded from the reference pool to avoid ambiguous propagation. Very short prefixes are also ignored, while all proposed matches are manually reviewed before assignment.

The same methodology was evaluated for building names. However, building-prefix matching produced predominantly generic, ambiguous or incorrect candidate matches, with only a very small number of valid cases. Building-prefix propagation was therefore not retained in the final pipeline.

## 6.1. Projects
The validated project-prefix mappings assign developers to an additional 2,809 transactions, **increasing overall developer coverage from 61.96% to 63.23%.**

In [67]:
def split_name_prefix(name):
    """
    Split a normalized name at the first non-letter/non-space character.

    Returns:
    - prefix: text before the first separator/number/symbol
    - suffix: remaining text, if present
    """
    if pd.isna(name):
        return pd.Series([None, None])

    parts = re.split(r'[^A-Za-z\s]+', name, maxsplit=1)

    prefix = parts[0].strip()

    if len(parts) == 1:
        return pd.Series([prefix, None])

    return pd.Series([prefix, parts[1].strip()])

In [68]:
def find_prefix_similarity_matches(
    df,
    prefix_col,
    cutoff=90,
    min_length=3
):
    """
    Match unresolved prefixes against prefixes with an identified developer.

    - Ignores empty/very short prefixes.
    - Excludes known prefixes associated with multiple developers.
    - Uses token_sort_ratio for fuzzy similarity.
    """

    # Known prefix/developer combinations
    known = (
        df[
            df['developer'].notna()
            & df[prefix_col].notna()
            & df[prefix_col].str.strip().ne('')
            & df[prefix_col].str.len().ge(min_length)
        ]
        [[prefix_col, 'developer']]
        .drop_duplicates()
    )

    # Identify ambiguous known prefixes
    ambiguous_prefixes = (
        known.groupby(prefix_col)['developer']
        .nunique()
    )

    ambiguous_prefixes = ambiguous_prefixes[
        ambiguous_prefixes > 1
    ]

    # Keep only unambiguous known prefixes
    known_clean = (
        known[
            ~known[prefix_col].isin(ambiguous_prefixes.index)
        ]
        .drop_duplicates(subset=prefix_col)
    )

    # Currently unmatched prefixes
    unknown = (
        df[
            df['developer'].isna()
            & df[prefix_col].notna()
            & df[prefix_col].str.strip().ne('')
            & df[prefix_col].str.len().ge(min_length)
        ]
        [[prefix_col]]
        .drop_duplicates()
    )

    matches = []

    for value in unknown[prefix_col]:

        match = process.extractOne(
            value,
            known_clean[prefix_col],
            scorer=fuzz.token_sort_ratio,
            score_cutoff=cutoff
        )

        if match:
            matched_value, score, _ = match

            matched_dev = known_clean.loc[
                known_clean[prefix_col] == matched_value,
                'developer'
            ].iloc[0]

            matches.append({
                'unmatched_prefix': value,
                'matched_prefix': matched_value,
                'developer_suggested': matched_dev,
                'similarity': score
            })

    columns = [
        'unmatched_prefix',
        'matched_prefix',
        'developer_suggested',
        'similarity'
    ]

    matches_df = pd.DataFrame(matches, columns=columns)

    if not matches_df.empty:
        matches_df = (
            matches_df
            .sort_values('similarity', ascending=False)
            .reset_index(drop=True)
        )

    print(
        f"{len(matches_df)} unmatched prefixes have "
        f"similarity >= {cutoff} with known prefixes."
    )

    print(
        f"{len(ambiguous_prefixes)} ambiguous known prefixes "
        f"were excluded from matching."
    )

    return matches_df, ambiguous_prefixes

In [69]:
df_dev[['project_prefix', 'project_suffix']] = (
    df_dev['project'].apply(split_name_prefix)
)

In [70]:
project_prefix_matches, ambiguous_project_prefixes = (
    find_prefix_similarity_matches(
        df_dev,
        prefix_col='project_prefix',
        cutoff=90
    )
)

26 unmatched prefixes have similarity >= 90 with known prefixes.
2 ambiguous known prefixes were excluded from matching.


In [71]:
display(project_prefix_matches)

,unmatched_prefix,matched_prefix,developer_suggested,similarity
0,elite,elite,tameer holdings,100.000000
1,reef,reef,reef luxury development,100.000000
2,the residences,the residences,al habtoor,100.000000
3,jumeirah park,jumeirah park,nakheel,100.000000
4,beach walk residences,beach walk residences,imtiaz,100.000000
5,the autograph,the autograph,green group,100.000000
6,victory heights,victory heights,dubai sports city,100.000000
7,sidra,sidra,emaar,100.000000
8,arabian ranches iii,arabian ranches iii,emaar,100.000000
9,park lane,park lane,heilbronn,100.000000


In [72]:
excluded_project_prefixes = [
    'elle residences', 'laya residences', 'balqis residence',
    'maya townhouses', 'verdana', 'the residence', 'time', 
    'park lane', 'jumeirah park', 'the residences'
]

In [73]:
def apply_prefix_matches(
    df,
    matches_df,
    prefix_col,
    excluded_prefixes=None
):
    """
    Apply validated prefix-based developer assignments
    only to currently unmatched transactions.
    """

    if excluded_prefixes is None:
        excluded_prefixes = []

    excluded_prefixes = {
        value.lower()
        for value in excluded_prefixes
    }

    valid_matches = matches_df[
        ~matches_df['unmatched_prefix']
        .str.lower()
        .isin(excluded_prefixes)
    ].copy()

    mapping_dict = (
        valid_matches
        .set_index('unmatched_prefix')['developer_suggested']
        .to_dict()
    )

    mask = (
        df['developer'].isna()
        & df[prefix_col].isin(mapping_dict)
    )

    df.loc[mask, 'developer'] = (
        df.loc[mask, prefix_col]
        .map(mapping_dict)
    )

    print(
        f"{mask.sum():,} transactions assigned through "
        f"{prefix_col} propagation."
    )

    return df

In [74]:
df_dev = apply_prefix_matches(
    df_dev,
    project_prefix_matches,
    prefix_col='project_prefix',
    excluded_prefixes=excluded_project_prefixes
)

2,809 transactions assigned through project_prefix propagation.


In [75]:
# verify coverage after propagation
coverage = df_dev['developer'].notna().mean() * 100
print(f"Developer coverage after safe propagation: {coverage:.2f}%")

Developer coverage after safe propagation: 63.23%


# 7. Final Integration and Export
Following completion of the developer-resolution pipeline, the final developer feature is merged back into the main analytical dataset.

The original transaction_id is retained throughout the developer-extraction process and used as the record-level merge key. This provides a direct one-to-one relationship between the developer-enrichment dataset and the original transaction data, avoiding reliance on potentially duplicated project, building or composite text identifiers.

Before merging, the uniqueness of transaction_id is verified in both datasets. The final transaction_id–developer mapping is then merged into the original dataset using a one-to-one left join, preserving all original transactions while adding the newly engineered developer feature.

The enriched dataset is subsequently exported for use in the remaining exploratory, statistical and predictive modelling stages of the project.

In [77]:
developer_map = (
    df_dev[['transaction_id', 'developer']]
    .drop_duplicates(subset='transaction_id')
)

In [78]:
developer_conflicts = (
    df_dev.groupby('transaction_id')['developer']
    .nunique(dropna=True)
)

developer_conflicts = developer_conflicts[
    developer_conflicts > 1
]

developer_conflicts

Series([], Name: developer, dtype: int64)

In [79]:
developer_map['transaction_id'].is_unique

True

In [80]:
print("df transaction_id unique:", df['transaction_id'].is_unique)
print("df_dev transaction_id unique:", df_dev['transaction_id'].is_unique)

df transaction_id unique: True
df_dev transaction_id unique: True


In [81]:
df = df.merge(
    developer_map,
    on='transaction_id',
    how='left',
    validate='one_to_one'
)

In [83]:
print(f"Rows in final dataset: {len(df):,}")

print(
    f"Developer coverage: "
    f"{df['developer'].notna().mean() * 100:.2f}%"
)

print(
    f"Unique developers: "
    f"{df['developer'].nunique()}"
)

Rows in final dataset: 220,918
Developer coverage: 63.23%
Unique developers: 148


In [84]:
# Saving the final dataset
df.to_csv("real_estate_2025.csv", index=False)